In [1]:
import pandas as pd
import re
import string

In [ ]:
data = pd.read_csv('dataset\dataYangDiPakai\data_sound_horeg_total_mei.csv')
data.head(20)

In [ ]:

# 1. Pastikan 'favorite_count' TIDAK ada di daftar kolom yang dibuang
kolomDibuang = [
    'conversation_id_str', 'id_str', 'image_url', 'in_reply_to_screen_name',
    'lang', 'location', 'quote_count', 'reply_count', 'retweet_count',
    'tweet_url', 'user_id_str', 'username'
]

# 2. Hapus kolom sampah
dataPembersihan = data.drop(columns=kolomDibuang)

# 3. Hapus baris kosong agar tidak error
dataPembersihan = dataPembersihan.dropna(subset=['full_text'])

# 4. Fungsi Pembersihan (Tetap sama)
def cleaningKhusus(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#','', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    text = text.strip()
    return text

def cleaningUmum(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = ' '.join([word for word in text.split() if len(word) > 1])
    return text

# 5. BERSIHKAN TEKS dan langsung simpan kembali ke kolom 'full_text'
# Ini akan menimpa teks asli dengan teks yang sudah bersih
dataPembersihan['full_text'] = dataPembersihan['full_text'].apply(cleaningKhusus).apply(cleaningUmum)

# 6. Pilih hanya 3 kolom yang Anda minta
df_hasil_akhir = dataPembersihan[['created_at', 'favorite_count', 'full_text']]

# 7. Simpan ke CSV
df_hasil_akhir.to_csv('dataset\dataYangDiPakai\data_sound_horeg_total_mei_dibersihkan.csv', index=False)

# Tampilkan hasil
print("Berhasil! Kolom sekarang hanya: created_at, favorite_count, dan full_text (sudah bersih).")
print(df_hasil_akhir.head(20))

In [2]:
pd.set_option('display.max_colwidth', None)

In [ ]:
df = pd.read_csv('dataset\dataYangDiPakai\hasil_preprocessing_intoleransi.csv')
df.head(20)

In [ ]:
#membersihkan data
import pandas as pd
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory, StopWordRemover, ArrayDictionary
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# --- PENGATURAN TAMPILAN ---
pd.set_option('display.max_colwidth', None)

# 1. LOAD DATA
# Path file sesuai dengan yang kamu berikan
filename = '/content/drive/MyDrive/proyek skiprsi sound horeg/hasil_data_bersih_lengkap2.csv'

try:
    df = pd.read_csv(filename)
    print(f"Data berhasil dimuat: {len(df)} baris")
except FileNotFoundError:
    print("Error: File tidak ditemukan. Pastikan path di Google Drive sudah benar.")
    # Dummy data untuk antisipasi error jika dijalankan orang lain
    df = pd.DataFrame({'full_text': ['sound horeg jancok gak enak', 'asu tenan', 'saya tidak setuju']})

# Hapus Duplikat
df = df.drop_duplicates(subset=['full_text'])

# 2. DEFINISI KAMUS (Kamus Alay + Istilah Intoleransi + Bahasa Jawa)
kamus_alay = {
    # --- A. KATA GANTI & SINGKATAN UMUM ---
    'org': 'orang', 'yg': 'yang', 'jg': 'juga', 'ga': 'tidak', 'udh': 'sudah',
    'jatim': 'jawa timur', 'bgt': 'banget', 'wong': 'orang', 'tak': 'tidak',
    'utk': 'untuk', 'trs': 'terus', 'gak': 'tidak', 'tu': 'itu', 'gimana': 'bagaimana',
    'sampe': 'sampai', 'ampe': 'sampai', 'jd': 'jadi', 'gw': 'aku', 'tau': 'tahu',
    'gara': 'karena', 'trus': 'terus', 'sm': 'sama', 'pake': 'pakai', 'klo': 'kalau',
    'gue': 'aku', 'tp': 'tapi', 'dr': 'dari', 'jgn': 'jangan', 'fasum': 'fasilitas umum',
    'gini': 'ini', 'ama': 'sama', 'knp': 'kenapa', 'cm': 'cuma', 'udah': 'sudah',
    'gada': 'tidak ada', 'gmn': 'bagaimana', 'emg': 'memang', 'krn': 'karena',
    'sdh': 'sudah', 'aja': 'saja', 'dlm': 'dalam', 'blm': 'belum', 'dgn': 'dengan',
    'scr': 'secara', 'adlh': 'adalah', 'tdk': 'tidak', 'skrg': 'sekarang',
    'bkn': 'bukan', 'sbg': 'sebagai', 'kalo': 'kalau', 'buanter': 'kencang',

    # --- B. ISTILAH KONFLIK & INTOLERANSI ---
    'brisik': 'berisik', 'bising': 'berisik', 'budeg': 'tuli', 'brebeken': 'berisik',
    'pekok': 'bodoh', 'goblok': 'bodoh', 'tolol': 'bodoh', 'edan': 'gila',
    'gendeng': 'gila', 'stress': 'gila', 'rusuh': 'rusak', 'ancur': 'hancur',
    'bakar': 'bakar', 'matek': 'mati', 'modar': 'mati', 'sampah': 'buruk',
    'sdm': 'sumber daya manusia', 'rendah': 'buruk', 'norak': 'kampungan',
    'ganggu': 'mengganggu', 'keganggu': 'terganggu',

    # --- C. BAHASA JAWA TIMURAN & KATA KASAR (UPDATED) ---
    'nek': 'kalau', 'iso': 'bisa', 'ra': 'tidak', 'ora': 'tidak', 'ae': 'saja',
    'wae': 'saja', 'akeh': 'banyak', 'seng': 'yang', 'sing': 'yang', 'wes': 'sudah',
    'wis': 'sudah', 'urung': 'belum', 'durung': 'belum', 'lapo': 'kenapa',
    'opo': 'apa', 'iki': 'ini', 'kuwi': 'itu', 'kae': 'itu', 'elek': 'jelek',
    'apik': 'bagus', 'onok': 'ada', 'karo': 'sama',

    # PERBAIKAN DI SINI:
    'cok': 'jancok',      # Singkatan disamakan ke jancok
    'dancok': 'jancok',   # Varian disamakan ke jancok
    # 'jancok' KITA BIARKAN (TIDAK ADA DI KAMUS) AGAR TIDAK BERUBAH

    'asu': 'anjing',      # Hewan tetap diterjemahkan

    # --- D. PERBAIKAN TYPO ---
    'gabisa': 'tidak bisa', 'gaenak': 'tidak enak', 'soundhoreg': 'sound horeg',
    'soundsystem': 'sound system', 'gasemua': 'tidak semua',

    # --- E. PENGHAPUSAN (Kata tanpa makna) ---
    'wkwkwkw': '', 'wkwkw': '', 'wkwkwkwk': '', 'wkwk': '', 'sih': '', 'nya': ''
}

# 3. PERSIAPAN SASTRAWI (MODIFIKASI KHUSUS INTOLERANSI)
factory_stop = StopWordRemoverFactory()
stopwords_list = factory_stop.get_stop_words()

# WHITELIST: Kata yang HARAM dihapus
whitelist = [
    'tidak', 'enggak', 'bukan', 'jangan', 'tapi',
    'masalah', 'kurang', 'belum', 'tak', 'tanpa',
    'soal', 'sebab', 'karena', 'akibat', 'padahal'
]

# Hapus whitelist dari daftar stopword bawaan
for word in whitelist:
    if word in stopwords_list:
        stopwords_list.remove(word)

# Tambahkan stopword sampah
stopwords_list.extend(['min', 'kak', 'gan', 'sis', 'guys', 'halo', 'hai'])

dictionary = ArrayDictionary(stopwords_list)
stopword_remover = StopWordRemover(dictionary)

factory_stem = StemmerFactory()
stemmer = factory_stem.create_stemmer()

# 4. FUNGSI PEMBERSIH UTAMA
def clean_text_complete(text):
    if not isinstance(text, str):
        return ""

    # A. Case Folding
    text = text.lower()

    # B. Ganti simbol dengan spasi
    text = re.sub(r'[^a-zA-Z0-9]', ' ', text)

    # C. Normalisasi Kata
    words = text.split()
    normalized_words = []
    for w in words:
        if w in kamus_alay:
            # Jika ada di kamus, ganti. Jika replacement '', kata dihapus.
            if kamus_alay[w] != '':
                normalized_words.append(kamus_alay[w])
        else:
            # Jika tidak ada di kamus (misal: 'jancok'), biarkan apa adanya
            normalized_words.append(w)

    text = ' '.join(normalized_words)

    # D. Stopword Removal
    text = stopword_remover.remove(text)

    # E. Stemming
    text = stemmer.stem(text)

    # F. Rapikan Spasi
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# 5. EKSEKUSI
print("Sedang memproses teks... (Mohon tunggu)")
df['full_text_clean'] = df['full_text'].apply(clean_text_complete)

# 6. SIMPAN HASIL
columns_to_save = ['created_at', 'favorite_count', 'full_text_clean']
valid_columns = [col for col in columns_to_save if col in df.columns]
df_final = df[valid_columns]

# Rename kolom hasil bersih menjadi 'full_text' agar siap dipakai
df_final = df_final.rename(columns={'full_text_clean': 'full_text'})

# Hapus baris kosong
df_final = df_final[df_final['full_text'].str.strip() != '']

# Simpan
output_file = 'hasil_preprocessing_intoleransi_final.csv'
df_final.to_csv(output_file, index=False)

print("\n--- SELESAI! ---")
print(f"File siap disimpan sebagai: {output_file}")
print("\nContoh Hasil (5 baris pertama):")
print(df_final.head())

In [ ]:
#labeling data ke 3 kategori

import pandas as pd
# 1. MOUNT DRIVE
# 2. LOAD DATA BERSIH
# Pastikan path file benar
input_filename = '/content/drive/MyDrive/proyek skiprsi sound horeg/hasil_preprocessing_intoleransi_final.csv'
df = pd.read_csv(input_filename)

# 3. DEFINISI KATA KUNCI (3 KATEGORI)

# KATA KUNCI RASIONAL (Label 1)
# Fokus: Dampak fisik, gangguan situasi, penolakan logis
keywords_rasional = [
    'ganggu', 'bising', 'berisik', 'brisik', 'polusi', 'tuli', 'budeg',
    'pecah', 'getar', 'runtuh', 'rusak', 'macet', 'blokir', 'tutup jalan',
    'sakit', 'pusing', 'jantung', 'bayi', 'orang tua', 'anak', 'nangis',
    'tidak setuju', 'tidak suka', 'tolak', 'keberatan', 'komplain',
    'aturan', 'izin', 'waktu', 'jam', 'solusi', 'saran', 'uang', 'biaya',
    'sebab', 'karena', 'gara', 'akibat', 'dampak', 'bikin', 'buat',
    'tidur', 'istirahat', 'belajar', 'ibadah', 'sholat', 'ngaji'
]

# KATA KUNCI CACI MAKI (Label 2)
# Fokus: Hinaan personal, hewan, kotoran, ancaman kosong
keywords_cacian = [
    'jancok', 'cok', 'dancok', 'asu', 'anjing', 'bangsat', 'bajingan',
    'goblok', 'tolol', 'pekok', 'bodoh', 'bego', 'idiot', 'setan', 'iblis',
    'sakit jiwa', 'gila', 'edan', 'gendeng', 'sdm rendah', 'kampungan',
    'bakar', 'musnah', 'usir', 'mati', 'modar', 'sampah', 'norak',
    'jelek', 'buruk'
]

# 4. FUNGSI LABELING 3 KATEGORI
def auto_label_3_class(text):
    if not isinstance(text, str):
        return 0 # Default Netral

    text_check = f" {text} "

    # PRIORITAS 1: Apakah ini Kritik Rasional? (Label 1)
    # Aturan: Walaupun kasar, kalau ada poin rasional, masuk sini.
    for word in keywords_rasional:
        if f" {word} " in text_check:
            return 1

    # PRIORITAS 2: Apakah ini Murni Caci Maki? (Label 2)
    # Aturan: Kasar tapi tidak ada alasan jelas.
    for word in keywords_cacian:
        if f" {word} " in text_check:
            return 2

    # PRIORITAS 3: Sisanya adalah Netral/Info (Label 0)
    return 0

# 5. EKSEKUSI
print("Sedang melabeli data menjadi 3 Kategori...")
print("0: Netral | 1: Kritik Rasional | 2: Caci Maki")
df['label'] = df['full_text'].apply(auto_label_3_class)

# 6. CEK HASIL SEBARAN
counts = df['label'].value_counts().sort_index()
print("\n--- Statistik Label Sementara ---")
print(f"Label 0 (Netral/Info)      : {counts.get(0, 0)} data")
print(f"Label 1 (Kritik Rasional)  : {counts.get(1, 0)} data")
print(f"Label 2 (Caci Maki Murni)  : {counts.get(2, 0)} data")

# 7. SIMPAN KE FILE BARU
output_file = '/content/drive/MyDrive/proyek skiprsi sound horeg/data_label_3_kategori.csv'
df.to_csv(output_file, index=False)
print(f"\n[SUKSES] File siap diverifikasi manual: {output_file}")